# Fixed-Speaker Bonafide Feasibility Check

This notebook checks whether the fixed speaker IDs chosen from the spoof runs have enough `bonafide` examples in the prepared ASVspoof5 manifests.

Target question:
- can we reuse the same train/dev speaker sets for bonafide TCAV runs with `20` utterances per speaker on the `test` split?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid')

In [ ]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
PLAN_BASE = (
    PROJECT_ROOT
    / 'data'
    / 'datasets'
    / 'ASVspoof5_tars'
    / 'ASVspoof5_protocols'
    / 'train_dev_16_systems_outputs'
)

SPLIT_NAME = 'test'
TARGET_CLASS = 'bonafide'
REQUIRED_UTTS_PER_SPEAKER = 20

TRAIN_SPEAKERS = [
    'T_0380', 'T_0411', 'T_0635', 'T_0897', 'T_1864',
    'T_2149', 'T_2284', 'T_2326', 'T_2791', 'T_3455',
    'T_3714', 'T_3850', 'T_3883', 'T_4049', 'T_4126',
    'T_4175', 'T_4618', 'T_4769', 'T_4913', 'T_5053',
]

DEV_SPEAKERS = [
    'D_0430', 'D_0461', 'D_0546', 'D_0956', 'D_2288',
    'D_2884', 'D_2937', 'D_2975', 'D_3192', 'D_3501',
    'D_3668', 'D_3927', 'D_3964', 'D_4023', 'D_4057',
    'D_4814', 'D_4825', 'D_4888', 'D_5112', 'D_5248',
]

MANIFEST_PATHS = {
    'train': PLAN_BASE / 'train' / 'selected_utterances_plan.csv',
    'dev': PLAN_BASE / 'dev' / 'selected_utterances_plan.csv',
}

for part, path in MANIFEST_PATHS.items():
    print(part, path, '| exists =', path.exists())

In [ ]:
# ===== Load manifests =====
manifest_by_partition = {}
required_cols = {
    'split', 'speaker_id', 'utt_id', 'gender', 'label',
    'system_id', 'sample_class', 'target_class'
}

for part, path in MANIFEST_PATHS.items():
    assert path.exists(), f'Missing manifest: {path}'
    df = pd.read_csv(path)
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{part} manifest missing columns: {sorted(missing)}')
    manifest_by_partition[part] = df.copy()
    print(part, 'rows =', len(df))

In [ ]:
# ===== Helper =====
def bonafide_speaker_table(df: pd.DataFrame, partition: str, speakers: list[str]) -> pd.DataFrame:
    sub = df[
        df['split'].eq(SPLIT_NAME)
        & df['target_class'].eq(TARGET_CLASS)
        & df['speaker_id'].isin(speakers)
    ].copy()

    counts = (
        sub.groupby('speaker_id')
        .agg(
            bonafide_rows=('utt_id', 'size'),
            bonafide_unique_utts=('utt_id', 'nunique'),
            gender=('gender', 'first'),
        )
        .reset_index()
    )

    base = pd.DataFrame({'speaker_id': speakers})
    merged = base.merge(counts, on='speaker_id', how='left')
    merged['partition'] = partition
    merged['bonafide_rows'] = merged['bonafide_rows'].fillna(0).astype(int)
    merged['bonafide_unique_utts'] = merged['bonafide_unique_utts'].fillna(0).astype(int)
    merged['enough_for_20'] = merged['bonafide_rows'] >= REQUIRED_UTTS_PER_SPEAKER
    merged = merged[['partition', 'speaker_id', 'gender', 'bonafide_rows', 'bonafide_unique_utts', 'enough_for_20']]
    return merged.sort_values(['enough_for_20', 'bonafide_rows', 'speaker_id'], ascending=[True, True, True]).reset_index(drop=True)

In [ ]:
# ===== Build tables =====
train_table = bonafide_speaker_table(manifest_by_partition['train'], 'train', TRAIN_SPEAKERS)
dev_table = bonafide_speaker_table(manifest_by_partition['dev'], 'dev', DEV_SPEAKERS)

display(train_table)
display(dev_table)

In [ ]:
# ===== Summary =====
summary_df = pd.DataFrame([
    {
        'partition': 'train',
        'n_requested_speakers': len(TRAIN_SPEAKERS),
        'n_with_at_least_20_bonafide': int(train_table['enough_for_20'].sum()),
        'min_bonafide_rows': int(train_table['bonafide_rows'].min()),
        'median_bonafide_rows': float(train_table['bonafide_rows'].median()),
        'max_bonafide_rows': int(train_table['bonafide_rows'].max()),
    },
    {
        'partition': 'dev',
        'n_requested_speakers': len(DEV_SPEAKERS),
        'n_with_at_least_20_bonafide': int(dev_table['enough_for_20'].sum()),
        'min_bonafide_rows': int(dev_table['bonafide_rows'].min()),
        'median_bonafide_rows': float(dev_table['bonafide_rows'].median()),
        'max_bonafide_rows': int(dev_table['bonafide_rows'].max()),
    },
])
display(summary_df)

In [ ]:
# ===== Missing / insufficient speakers =====
insufficient_train = train_table[~train_table['enough_for_20']].copy()
insufficient_dev = dev_table[~dev_table['enough_for_20']].copy()

display(insufficient_train)
display(insufficient_dev)

In [ ]:
# ===== Distribution plots =====
plot_df = pd.concat([
    train_table[['partition', 'speaker_id', 'bonafide_rows']],
    dev_table[['partition', 'speaker_id', 'bonafide_rows']],
], axis=0, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)

for ax, part in zip(axes, ['train', 'dev']):
    sub = plot_df[plot_df['partition'].eq(part)].copy().sort_values('bonafide_rows')
    sns.barplot(data=sub, x='speaker_id', y='bonafide_rows', ax=ax, color='#4C78A8')
    ax.axhline(REQUIRED_UTTS_PER_SPEAKER, color='crimson', linestyle='--', linewidth=1.5)
    ax.set_title(f'{part}: bonafide rows for fixed speakers')
    ax.set_xlabel('speaker_id')
    ax.set_ylabel('bonafide rows')
    ax.tick_params(axis='x', rotation=90)

plt.show()

## Decision Rule

If all `train` and all `dev` speakers satisfy `enough_for_20 == True`, then these fixed speaker lists can be reused for matched bonafide TCAV runs with `20` test utterances per speaker.

If some speakers fail, then the next step is to either:
- reduce the bonafide utterance target below `20`, or
- replace the failing speakers with alternatives that satisfy the threshold.